# Lesson 07 · Momentum, Adam & AdamW: racing three optimizers, from a narrow valley to an MNIST network

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/07-bo-toi-uu-en.ipynb)

This notebook accompanies [Lesson 07 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/bo-toi-uu/), a deep-learning course originally in
Vietnamese. It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/07-bo-toi-uu.ipynb)):
the code, the data and every check are the same. The interactive demo is on the lesson page.
Gradient descent in Lesson 05 had only one knob, the learning rate $\eta$. Here we write three optimizers ourselves
in numpy, **SGD, Momentum and Adam**, race them on the demo's narrow valley and then on the course's
784 → 128 → 64 → 10 MNIST network, and see what warmup, the cosine schedule and AdamW change.

**What you'll do:**
1. See why SGD zig-zags in a narrow valley, and why $\eta$ is capped by the steepest direction.
2. Write Momentum and Adam (with bias correction), race them against SGD, then tilt the valley by 20°.
3. Race the three on a real MNIST network, reproduce the demo's curves exactly, then read the results of a sweep
   over $\eta$ × 3 seeds.
4. Use warmup + cosine to train stably with $\eta = 1$, a value at which a fixed $\eta$ blows up.
5. Compare Adam + an L2 penalty with AdamW on weights that never receive a gradient.

**Background:** [Lesson 05 · Gradient descent](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/gradient-descent/)
(learning rate, mini-batches) and [Lesson 06 · Backpropagation](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/lan-truyen-nguoc/)
(the gradient of an MLP, reused in section 6). **Run time:** under a minute on a CPU, including the first MNIST
download (~11 MB).

Every number in the notebook is computed by the cells below, and matches the numbers in the demo. The one exception
is the full sweep (84 training runs of 5 epochs), whose results are loaded from the very file the demo uses, because
rerunning it takes a few minutes; the notebook reruns part of it and checks that it gets the same numbers.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

Only numpy and matplotlib are needed (Colab has them preinstalled). The function `num()` prints numbers in the
English format: a decimal point, and commas between thousands. That is the same point Python uses when you *type* a
decimal (`0.1`); printing only adds the thousands separators and a real minus sign.

In [ ]:
import gzip
import hashlib
import itertools
import json
import math
import urllib.request
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter

# The course's palette (colorblind-safe), always paired with a second encoding (marker shape, dash style, label)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# Each optimizer has its own color, marker shape and dash style, as in the demo
STYLE = {
    "SGD": {"color": BLUE, "marker": "o", "ls": "-"},
    "Momentum": {"color": ORANGE, "marker": "s", "ls": (0, (6, 3))},
    "Adam": {"color": GREEN, "marker": "^", "ls": (0, (1.2, 2.2))},
}


def num(x, digits=2):
    """A number in the English format: 13,007 · 0.1 · −4.38."""
    return f"{x:,.{digits}f}".replace("-", "−")


def pct(p, digits=2):
    return num(100 * p, digits) + "%"


def num_eta(e):
    """η in the tables, with one significant digit: 0.3 · 0.03 · 0.001 · 1."""
    return num(e, max(0, -math.floor(math.log10(e) + 1e-9)))


def tick(v, _pos=None):
    """An axis tick label: 0.5 · −2 · 0.001 (no trailing zeros)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def fmt_axes(fig):
    """Put the tick labels of every axis of the figure in the English format. Log axes get 1–2–5 ticks, or powers of
    10 if the axis is very long."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() == "log":
                lo, hi = sorted(lim)
                mult = (1, 2, 5) if hi / lo < 300 else (1,)
                axis.set_major_locator(
                    FixedLocator([m * 10.0**k for k in range(-6, 4) for m in mult if lo <= m * 10.0**k <= hi])
                )
                axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(tick))


print(num(13007, 0), num(0.1, 1), num(-4.3805, 4), pct(0.993527), num_eta(0.003), tick(0.05), tick(-2.0))

## 2 · The data

MNIST (LeCun et al., 1998): 60,000 training images and 10,000 test images, grayscale 28×28 handwritten digits.
The cell below downloads the 4 original files and checks their MD5 hashes, so everyone who runs it gets exactly the
same data.

As in the demo, we split them into three parts, each with one job:
- **training:** the first 55,000 images, for computing gradients;
- **validation:** the last 5,000 images of the training file, for *choosing* $\eta$;
- **test:** the 10,000 images of the test file, only for reporting the final result, not for choosing anything.

The cell also loads the file of sweep results that the demo uses (`fetch_json`, with an MD5 check).

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # MNIST's standard MD5 hashes
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"


def load(kind):
    """IDX format: the first 16 bytes of an image file (8 bytes of a label file) are the header; the rest are bytes
    0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 28, 28)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# The results of a long training run (the same file as the demo on the web). build.py replaces COURSE_DATA_URL with
# the data/ folder of the public notebook repo.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
X_all, y_all = train_img.reshape(-1, 784) / 255.0, train_lab.astype(np.int64)
Xtr, ytr = X_all[:55000], y_all[:55000]  # training
Xval, yval = X_all[55000:], y_all[55000:]  # validation: η is chosen here
Xte, yte = test_img.reshape(-1, 784) / 255.0, test_lab.astype(np.int64)  # test: for reporting only
runs = fetch_json("mnist-optimizers.json", "c4835a5fec46888af1bd11bcab00a947")  # the demo's sweep results
print(f"training {num(len(ytr), 0)} · validation {num(len(yval), 0)} · test {num(len(yte), 0)} images")
n_cfg, n_seed = len(runs["sweep"]), len(runs["seeds"])
print(
    f"sweep file: {n_cfg} configurations (optimizer × η × schedule) × {n_seed} seeds = {n_cfg * n_seed} runs of 5 epochs"
)
assert (len(ytr), len(yval), len(yte)) == (runs["nTrain"], runs["nVal"], runs["nTest"])

One image of each digit, taken from the validation set:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(Xval[np.flatnonzero(yval == d)[0]].reshape(28, 28), cmap="gray_r")
    ax.set_title(str(d), fontsize=11)
    ax.axis("off")
fmt_axes(fig)
plt.show()

## 3 · SGD in a narrow valley

Before the real network, we use the demo's valley, a function of two variables that we know everything about:

$$L(\theta) = \tfrac12\left(0.04\,\theta_1^2 + 4\,\theta_2^2\right) = 0.02\,\theta_1^2 + 2\,\theta_2^2$$

The gradient is $g = (0.04\,\theta_1,\ 4\,\theta_2)$: the slope along $\theta_2$ is 100 times steeper than along
$\theta_1$. The two numbers 0.04 and 4 are the **curvatures** in the two directions (the eigenvalues of the Hessian
matrix). The loss of a neural network also has very steep directions and very gentle ones like these, only in
hundreds of thousands of dimensions.

Each optimizer is a function `step(theta, g, s, eta, t)`: it receives the gradient `g` at step `t`, updates `theta`
in place, and keeps its memory in the dictionary `s`. SGD (Lesson 05) needs no memory:
$\theta \leftarrow \theta - \eta\, g$.

The cell below also defines `race()`: it runs an optimizer for 200 steps from the point $(-5.5,\ 2)$, as the demo
does. “Arriving” means $L < 0.01$ and staying there until the end; “blowing up” is when $|\theta|$ exceeds 1,000.

In [ ]:
LAM = np.array([0.04, 4.0])  # the curvature along the valley floor and across the valley
TILT = math.radians(20)  # the tilt angle used in section 5
START = np.array([-5.5, 2.0])
GOAL, N_STEPS, BOOM = 0.01, 200, 1e3


def rot(p, a):
    c, s = math.cos(a), math.sin(a)
    return np.array([c * p[0] - s * p[1], s * p[0] + c * p[1]])


def valley_loss(theta, tilt=0.0):
    u = rot(theta, -tilt)  # the coordinates in the valley's own axes
    return 0.5 * float((LAM * u**2).sum())


def valley_grad(theta, tilt=0.0):
    return rot(LAM * rot(theta, -tilt), tilt)


def sgd(theta, g, s, eta, t):
    theta -= eta * g


def race(step, eta, start=START, tilt=0.0, n=N_STEPS):
    """Run n steps. Return the path, the losses, the blow-up step (or None) and the arrival step (or None)."""
    theta, s = np.array(start, float), {}
    path, loss = [theta.copy()], [valley_loss(theta, tilt)]
    for t in range(1, n + 1):
        step(theta, valley_grad(theta, tilt), s, eta, t)
        path.append(theta.copy())
        loss.append(valley_loss(theta, tilt))
        if np.hypot(*theta) > BOOM:
            return np.array(path), np.array(loss), t, None
    bad = np.flatnonzero(np.array(loss) >= GOAL)  # arrival: from this step to the end, every loss is below GOAL
    arrive = None if len(bad) and bad[-1] == n else (int(bad[-1]) + 1 if len(bad) else 0)
    return np.array(path), np.array(loss), None, arrive


def outcome(r):
    return (
        f"blows up at step {r[2]}" if r[2] else (f"arrives at step {r[3]}" if r[3] is not None else "not arrived yet")
    )


r_sgd = race(sgd, 0.45)
d = r_sgd[0][1] - START
print(f"SGD, η = 0.45: {outcome(r_sgd)}; first step Δθ = ({num(d[0], 3)}, {num(d[1], 3)})")
assert r_sgd[3] == 113, "differs from the demo: check the valley"

The first step goes almost entirely along $\theta_2$: the gradient is far larger in the steep direction, so SGD
dashes across to the opposite wall. The figure below draws the path on the contour lines (left; the + is the bottom,
the dotted ellipse is the goal region), and the two coordinates step by step (right): $\theta_2$ (gray, squares)
changes sign at every step, while $\theta_1$ (black, circles) crawls slowly toward 0.

In [ ]:
g1, g2 = np.meshgrid(np.linspace(-6.5, 2.5, 361), np.linspace(-4.0, 2.5, 261))


def draw_valley(ax, tilt=0.0, title=""):
    u1, u2 = rot(np.stack([g1, g2]), -tilt)  # rot() also works on the whole grid
    Lg = 0.5 * (LAM[0] * u1**2 + LAM[1] * u2**2)
    ax.contour(g1, g2, Lg, levels=[0.05, 0.2, 0.5, 1, 2, 4, 8], colors=GREY, linewidths=0.7)
    ax.contour(g1, g2, Lg, levels=[GOAL], colors="black", linewidths=1, linestyles=":")
    ax.plot(0, 0, "+", color="black", ms=10, mew=1.5)
    s0 = rot(START, tilt)
    ax.plot(s0[0], s0[1], "o", mfc="none", mec="black", ms=10, mew=1.5)
    ax.set(xlim=(-6.5, 2.5), ylim=(-4.0, 2.5), xlabel="θ₁", ylabel="θ₂", title=title)
    ax.set_aspect("equal")


def draw_path(ax, path, name, n=60, every=1):
    st = STYLE[name]
    p = path[: n + 1]
    ax.plot(
        p[:, 0],
        p[:, 1],
        color=st["color"],
        ls=st["ls"],
        lw=1.4,
        marker=st["marker"],
        ms=3.5,
        markevery=every,
        label=name,
    )


fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), gridspec_kw={"width_ratios": [1.5, 1]})
draw_valley(a, title="SGD, η = 0.45: the first 60 steps")
draw_path(a, r_sgd[0], "SGD")
k = np.arange(61)
b.plot(k, r_sgd[0][:61, 0], color="black", marker="o", ms=3, lw=1.2, label="θ₁ (along the floor)")
b.plot(k, r_sgd[0][:61, 1], color=GREY, marker="s", ms=3, lw=1.2, ls=(0, (6, 3)), label="θ₂ (across the valley)")
b.axhline(0, color=GREY, lw=0.8)
b.set(xlabel="step", ylabel="coordinate", title="The two coordinates, step by step")
b.legend(frameon=False, fontsize=9)
fmt_axes(fig)
plt.tight_layout()
plt.show()

Why does this happen? Along $\theta_2$, one SGD step is
$\theta_2 \leftarrow \theta_2 - \eta \cdot 4\theta_2 = (1 - 4\eta)\,\theta_2$; along $\theta_1$ it is
$\theta_1 \leftarrow (1 - 0.04\,\eta)\,\theta_1$. Each direction gets multiplied by a fixed factor at every step:

- $1 - 4\eta < 0$ when $\eta > 0.25$: $\theta_2$ changes sign at every step, that is, it bounces back and forth
  between the two walls.
- $|1 - 4\eta| > 1$ when $\eta > 0.5$: each bounce goes farther than the one before, and SGD blows up.
- Along $\theta_1$, the factor $1 - 0.04\,\eta$ is very close to 1, so along the floor SGD crawls very slowly.

The steepest direction sets the limit on $\eta$; the gentlest direction sets the speed. The cell below tries three
values of $\eta$:

In [ ]:
for eta in [0.25, 0.45, 0.55]:
    r = race(sgd, eta)
    print(
        f"η = {num(eta)}: θ₂ ×{num(1 - 4 * eta, 2):>6} per step, θ₁ ×{num(1 - 0.04 * eta, 3)} per step → {outcome(r)}"
    )
print(f"condition number κ = 4 / {num(LAM[0])} = {num(LAM[1] / LAM[0], 0)}")

With $\eta = 0.25$, SGD doesn't bounce, but 200 steps are still not enough for it to crawl to the goal; $\eta = 0.55$
blows up. No $\eta$ is both bounce-free and fast. The ratio between the largest and the smallest curvature (here
4 / 0.04 = 100) is called the **condition number** $\kappa$; the larger $\kappa$, the slower SGD. The next two
optimizers fix this problem in two different ways.

## 4 · Momentum: a ball with inertia

Momentum (Polyak, 1964) keeps a **velocity** $v$ that accumulates the gradients seen so far:

$$v \leftarrow \beta\, v + g, \qquad \theta \leftarrow \theta - \eta\, v$$

This is PyTorch's form, and the demo's. (Here $v$ is the velocity; in Adam, section 5, $v$ is a different
quantity.) Unrolling the formula: $v_t = g_t + \beta g_{t-1} + \beta^2 g_{t-2} + \dots$. The older a gradient, the
lighter its weight, by powers of $\beta$. In a direction where the gradient keeps changing sign (across the valley),
the terms cancel each other out. In a direction where the gradient keeps the same sign (along the floor), they add
up, to at most $1/(1-\beta)$ times:

In [ ]:
beta = 0.9
print("weights of g_t, g_{t−1}, g_{t−2}, …:", " · ".join(num(beta**k, 3) for k in range(6)), "…")
print(f"sum of all the weights = 1/(1 − β) = {num(1 / (1 - beta), 0)}")


def momentum(theta, g, s, eta, t, beta=0.9):
    s["v"] = beta * s.get("v", 0.0) + g  # velocity: the new gradient + β × the old velocity
    theta -= eta * s["v"]


r_mom = race(momentum, 0.45)
print(f"Momentum, η = 0.45, β = 0.9: {outcome(r_mom)} (SGD: {outcome(r_sgd)})")
print(f"largest θ₁ along the path: {num(r_mom[0][:, 0].max(), 2)} (starting from {num(START[0], 1)}, bottom at 0)")
assert r_mom[3] == 69, "differs from the demo: check the valley"

Momentum arrives at step 69, 44 steps sooner than SGD, but not in a straight line: its inertia carries it past the
bottom to $\theta_1 = 1.50$ before it turns back.

### 🤔 Predict first
The closer $\beta$ is to 1, the greater the inertia. With the same $\eta = 0.45$, does Momentum with $\beta = 0.99$
arrive sooner or later than with $\beta = 0.9$?

In [ ]:
betas = [0, 0.5, 0.75, 0.9, 0.95, 0.99]
for b_ in betas:
    print(f"β = {num(b_):>4}: {outcome(race(partial(momentum, beta=b_), 0.45))}")
beta_star = (1 - math.sqrt(0.45 * LAM[0])) ** 2
print(f"just enough damping in the gentle direction: β* = (1 − √(η·0.04))² = {num(beta_star, 3)}")
assert race(partial(momentum, beta=0), 0.45)[3] == r_sgd[3]  # β = 0 is exactly SGD

Much later: with $\beta = 0.99$ it still hasn't arrived after 200 steps. With too much inertia, the ball rolls back
and forth around the bottom for a long time before it stops. $\beta = 0$ coincides exactly with SGD, and the fastest
is $\beta$ around 0.75. That is the $\beta^*$ that gives “just enough damping” in the gentle direction: below it the
ball still crawls, above it it oscillates (Goh, *Why Momentum Really Works*, 2017). The $\beta = 0.9$ customary in
practice is not optimal for every problem.

Momentum also tolerates a larger $\eta$. In the steep direction it is stable when $\eta \cdot 4 < 2 + 2\beta$ (Goh,
2017), that is, $\eta < 0.95$ with $\beta = 0.9$, whereas SGD needs $\eta < 0.5$:

In [ ]:
r1, r2 = race(sgd, 0.6), race(momentum, 0.6)
print(f"η = 0.6: SGD {outcome(r1)}, Momentum {outcome(r2)}; Momentum's threshold (2 + 2β)/4 = {num((2 + 2 * 0.9) / 4)}")
assert r1[2] == 19 and r2[3] == 69

## 5 · Adam: a step size for each parameter

Momentum still uses one shared $\eta$ for every direction. Adam (Kingma & Ba, 2015) divides the step of each
parameter by the magnitude of the gradient of *that very parameter*. It keeps two moving averages, $m$ of $g$ and
$v$ of $g^2$, and steps by their ratio:

$$m \leftarrow \beta_1 m + (1-\beta_1)\, g \qquad v \leftarrow \beta_2 v + (1-\beta_2)\, g^2 \qquad
\theta \leftarrow \theta - \eta\, \frac{\hat m}{\sqrt{\hat v} + \epsilon}$$

with $\hat m = m/(1-\beta_1^t)$, $\hat v = v/(1-\beta_2^t)$, and the defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$,
$\epsilon = 10^{-8}$ (Algorithm 1 of the paper). Every operation is elementwise.

**Bias correction.** $m$ and $v$ start at 0, so in the first steps they are pulled toward 0: after step 1,
$m = 0.1\,g$ and $v = 0.001\,g^2$. Dividing by $1 - \beta^t$ corrects exactly that. Without it, the first step is
$0.1/\sqrt{0.001} \approx 3.16$ times $\eta$ long.

In [ ]:
def adam(theta, g, s, eta, t, b1=0.9, b2=0.999, eps=1e-8, correct=True):
    s["m"] = b1 * s.get("m", 0.0) + (1 - b1) * g  # moving average of g
    s["v"] = b2 * s.get("v", 0.0) + (1 - b2) * g * g  # moving average of g²
    m_hat = s["m"] / (1 - b1**t) if correct else s["m"]  # bias correction
    v_hat = s["v"] / (1 - b2**t) if correct else s["v"]
    theta -= eta * m_hat / (np.sqrt(v_hat) + eps)


g0 = valley_grad(START)
for correct in [True, False]:
    r = race(partial(adam, correct=correct), 0.2)
    d = r[0][1] - START
    print(
        f"Adam η = 0.2, {'with' if correct else 'without'} correction: first step Δθ = ({num(d[0], 3)}, {num(d[1], 3)}), {outcome(r)}"
    )
print(
    f"whereas the gradient at the starting point is g = ({num(g0[0], 2)}, {num(g0[1], 0)}): a factor of {num(g0[1] / -g0[0], 0)} apart"
)
print(f"without correction, step 1: m/√v = 0.1/√0.001 = {num(0.1 / math.sqrt(0.001), 2)} times η")
r_adam = race(adam, 0.2)
assert r_adam[3] == 61 and np.allclose(np.abs(r_adam[0][1] - START), 0.2, atol=1e-7)

Adam's first step is exactly $\eta = 0.2$ long along **both** axes, even though the gradient along $\theta_2$ is 36
times larger than along $\theta_1$. With bias correction, $\hat m_1 / \sqrt{\hat v_1} = g/|g| = \pm 1$: Adam's first
step uses only the *sign* of the gradient. From then on, the steep direction no longer caps Adam's $\eta$: it
arrives at step 61, sooner even than Momentum.

Adam doesn't always take steps of length $\eta$. $\sqrt{\hat v}$ remembers about 1,000 steps of gradients
($\beta_2 = 0.999$), $\hat m$ only about 10 steps. When the gradient changes sign (near the bottom), $\hat m$ shrinks
first; when the gradient gradually gets smaller, $\sqrt{\hat v}$ still remembers the old large gradients. Both make
the steps shorter. The cell below measures the step length on the valley, and on the one-variable function
$L = \theta^2/2$, where the gradient never changes sign:

In [ ]:
step_len = np.linalg.norm(np.diff(r_adam[0], axis=0), axis=1)
print("valley, |Δθ| at step 1 · 10 · 50 · 100:", " · ".join(num(step_len[k - 1], 4) for k in [1, 10, 50, 100]))
theta1, s1, steps1 = np.array([5.0]), {}, []
for t in range(1, 401):
    before = theta1.copy()
    adam(theta1, theta1.copy(), s1, 0.01, t)  # L = θ²/2, so g = θ
    steps1.append(float(before[0] - theta1[0]))
print(
    f"L = θ²/2 from θ = 5, η = 0.01: step 1 is {num(steps1[0], 4)} long, step 400 is {num(steps1[-1], 4)} long "
    f"({pct(steps1[-1] / 0.01, 0)} of η); θ still positive: {bool(theta1[0] > 0)}"
)
print(f"memory: β₁ = 0.9 → ~{num(1 / (1 - 0.9), 0)} steps, β₂ = 0.999 → ~{num(1 / (1 - 0.999), 0)} steps")

On the valley, the first step is 0.2828 = 0.2·√2 long (the diagonal of a step of ±0.2 along each axis), and near the
bottom only 0.0041: the gradient changes sign, so $\hat m$ cancels out. On $L = \theta^2/2$, the gradient never
changes sign once, and yet after 400 steps the step is only 57% of $\eta$. The reason is that $\sqrt{\hat v}$ still
remembers the large gradients of the beginning, while $\hat m$ has already caught up with the small current
gradient. So Adam's steps get shorter when the gradient is noisy, when it changes sign, and also when it shrinks
steadily.

### 🤔 Predict first
Tilt the valley by 20° (the same shape, the same starting point relative to the bottom; only the axes are rotated).
Which of the three optimizers changes its result, and in which direction?

In [ ]:
results = {}
for name, step, eta in [("SGD", sgd, 0.45), ("Momentum", momentum, 0.45), ("Adam", adam, 0.2)]:
    flat_r, tilted_r = race(step, eta), race(step, eta, start=rot(START, TILT), tilt=TILT)
    results[name] = (flat_r, tilted_r)
    print(f"{name:>8}: along the axes {outcome(flat_r):>20}  ·  tilted 20° {outcome(tilted_r)}")
assert [results[n][1][3] for n in results] == [113, 69, 168], "differs from the demo: check the valley"
d = np.diff(results["Adam"][1][0][:4], axis=0)
print(
    f"tilted 20°: the directions of Adam's first 3 steps are {' · '.join(num(a, 1) for a in np.degrees(np.arctan2(d[:, 1], d[:, 0])))} degrees"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for ax, j, title in [(axes[0], 0, "Along the axes"), (axes[1], 1, "Tilted 20°")]:
    draw_valley(ax, tilt=TILT * j, title=title)
    for name in results:
        draw_path(ax, results[name][j][0], name, n=N_STEPS, every=4)
axes[0].legend(frameon=False, fontsize=9, loc="lower right")
fmt_axes(fig)
plt.tight_layout()
plt.show()

SGD and Momentum give exactly the same results: their formulas use only the vector $g$, so rotating the axes
rotates the path along with them. Adam, though, is much slower, going from step 61 to step 168. It divides the
gradient by the separate magnitudes on the $\theta_1$ axis and on the $\theta_2$ axis, so it only helps when the steep
and the gentle directions lie along the axes, that is, along individual parameters. When the valley floor lies
diagonally, each parameter sees both the steep part and the gentle part. In the right-hand plot, Adam's first steps
(green, triangles) follow the −45° diagonal of the axes, because each coordinate steps only by the sign of its own
gradient, not by the shape of the valley.

A real network lies somewhere between these two cases. That's why the question “does Adam win?” has to be answered
by experiment.

## 6 · Racing on a real MNIST network

The course's network (Lessons 04–06): $h_1 = \mathrm{ReLU}(xW_1 + b_1)$, $h_2 = \mathrm{ReLU}(h_1W_2 + b_2)$,
$z = h_2W_3 + b_3$, softmax + cross-entropy, He initialization, batch size 64. The backpropagation is Lesson 06's.
The function `train()` below does exactly what `reel/mnist_mlp/optimizers.py`, the script that generates the demo's
data, does:
- the same seed gives the same initial weights and the same batch order, so the only difference left is the
  optimizer;
- each epoch is 860 steps, and the validation loss is measured after every ¼ epoch (215 steps);
- a run “blows up” when the loss of a batch is no longer a finite number.

In [ ]:
SIZES, BATCH, STEPS = [784, 128, 64, 10], 64, 4300  # 5 epochs × 860 steps, as in the demo


def forward(params, x):
    (W1, b1), (W2, b2), (W3, b3) = params
    h1 = np.maximum(0, x @ W1 + b1)
    h2 = np.maximum(0, h1 @ W2 + b2)
    return h1, h2, h2 @ W3 + b3


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def evaluate(params, X, y):
    z = forward(params, X)[2]
    p = softmax(z)
    return float(-np.log(p[np.arange(len(y)), y]).mean()), float((z.argmax(1) == y).mean())


def backward(params, x, y):
    """The batch loss, and the gradients in the order W1, b1, W2, b2, W3, b3 (Lesson 06)."""
    h1, h2, z = forward(params, x)
    p = softmax(z)
    loss = -np.log(p[np.arange(len(y)), y]).mean()
    dz = p.copy()
    dz[np.arange(len(y)), y] -= 1
    dz /= len(y)
    _, (W2, _), (W3, _) = params
    dh2 = (dz @ W3.T) * (h2 > 0)
    dh1 = (dh2 @ W2.T) * (h1 > 0)
    return float(loss), [x.T @ dh1, dh1.sum(0), h1.T @ dh2, dh2.sum(0), h2.T @ dz, dz.sum(0)]


def lr_at(eta, schedule, t, warmup=0.05, floor=0.1):
    """η at step t: fixed, or a linear warmup over 5% of the steps, then a cosine decay to 10% of η (section 8)."""
    if schedule == "fixed":
        return eta
    w = round(warmup * STEPS)
    if t <= w:
        return eta * t / w
    return eta * (floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * (t - w) / (STEPS - w))))


def train(step, eta, schedule="fixed", seed=0, n=STEPS, step_bias=None):
    """n steps of the optimizer `step` (the biases use `step_bias` if given). Returns the parameters, the log and the
    optimizer's memory."""
    rng = np.random.default_rng(seed)
    params = [[rng.normal(0, np.sqrt(2 / a), (a, b)), np.zeros(b)] for a, b in itertools.pairwise(SIZES)]  # He
    flat = [a for layer in params for a in layer]
    states, rules = [{} for _ in flat], [step, step_bias or step] * 3
    log, t = {"loss": [], "val": [evaluate(params, Xval, yval)], "boom": None}, 0
    with np.errstate(all="ignore"):  # a run that blows up overflows; we detect that ourselves below
        while t < n:
            order = rng.permutation(len(Xtr))
            for i in range(0, len(order), BATCH):
                idx = order[i : i + BATCH]
                loss, grads = backward(params, Xtr[idx], ytr[idx])
                t += 1
                log["loss"].append(loss)
                if not np.isfinite(loss):
                    log["boom"] = t
                    return params, log, states
                for a, g, s, rule in zip(flat, grads, states, rules):
                    rule(a, g, s, lr_at(eta, schedule, t), t)
                if t % 215 == 0:
                    log["val"].append(evaluate(params, Xval, yval))
                if t == n:
                    break
    return params, log, states


print(
    f"{num(sum(a * b + b for a, b in itertools.pairwise(SIZES)), 0)} parameters; {len(Xtr) // BATCH + 1} steps per epoch"
)

### 🤔 Predict first
The demo tried a few values of $\eta$ for each optimizer and kept the $\eta$ with the lowest validation loss after
5 epochs. For Momentum with $\beta = 0.9$ (the form $v \leftarrow \beta v + g$ from section 4), is the best $\eta$
larger or smaller than SGD's, and by roughly what factor?

In [ ]:
best = {(r["opt"], r["schedule"]): r for r in runs["sweep"] if r.get("best")}
NAMES = {"sgd": "SGD", "momentum": "Momentum", "adam": "Adam"}
for opt, name in NAMES.items():
    grid = [r["eta"] for r in runs["sweep"] if r["opt"] == opt and r["schedule"] == "fixed"]
    print(
        f"{name:>8}: tried η ∈ {{{', '.join(num_eta(e) for e in grid)}}} → best η = {num_eta(best[(opt, 'fixed')]['eta'])}"
    )
for sch in ["fixed", "cosine"]:
    ratio = best[("sgd", sch)]["eta"] / best[("momentum", sch)]["eta"]
    print(f"{'fixed' if sch == 'fixed' else 'warmup + cosine'} schedule: SGD's η / Momentum's η = {num(ratio, 0)}")
print(f"1/(1 − β) = {num(1 / (1 - runs['beta']), 0)}")

Smaller, by a factor of 10, with both schedules. In the form $v \leftarrow \beta v + g$, along a direction where the
gradient barely changes, $v$ approaches $g/(1-\beta) = 10\,g$, so Momentum with $\eta$ moves almost like SGD with
$10\eta$. Comparing the two at “the same $\eta$” is unfair. (The $\eta$ grid is sparse, each notch about 3 times the
one before, so this is “about 10”, not exactly 10.) Adam has its own $\eta$ scale: for it, $\eta$ is the step
length, not multiplied by the gradient.

Now we rerun the first epoch of the three best runs (seed 0), and compare it with the curves the demo saved:

In [ ]:
OPTS = {"SGD": ("sgd", sgd), "Momentum": ("momentum", momentum), "Adam": ("adam", adam)}
live = {}
for name, (key, step) in OPTS.items():
    eta = best[(key, "fixed")]["eta"]
    live[name] = train(step, eta, n=860)
    log = live[name][1]
    ref = next(c for c in runs["curves"] if c["opt"] == key and c["schedule"] == "fixed")
    got = [v for v, _ in log["val"]]
    smooth = np.array(log["loss"]).reshape(-1, 43).mean(1)  # the mean of every 43 steps, as in the demo
    print(
        f"{name:>8}, η = {num_eta(eta)}: validation after 0 · ¼ · ½ · ¾ · 1 epoch: {' · '.join(num(v, 4) for v in got)}"
        f"  →  {pct(log['val'][-1][1])} correct"
    )
    assert np.allclose(got, ref["val"][:5], atol=5e-4) and np.allclose(smooth, ref["train"][:20], atol=5e-4), (
        "differs from the demo: check the data"
    )
    assert abs(log["val"][-1][1] - ref["valAcc"][4]) < 0.001, "differs from the demo: check the data"
print("matches the seed-0 curves the demo saved (difference < 0.0005)")

The figure below plots the training loss (the mean of every 43 steps, log scale) over that one epoch. Each optimizer
has its own color, marker shape and dash style, as in the demo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
for name in OPTS:
    smooth = np.array(live[name][1]["loss"]).reshape(-1, 43).mean(1)
    st = STYLE[name]
    eta = best[(OPTS[name][0], "fixed")]["eta"]
    ax.plot(
        43 * np.arange(1, 21),
        smooth,
        color=st["color"],
        ls=st["ls"],
        marker=st["marker"],
        ms=4,
        lw=1.6,
        label=f"{name}, η = {num_eta(eta)}",
    )
ax.set_yscale("log")
ax.set(
    xlabel="step (1 epoch = 860 steps)",
    ylabel="training loss",
    title="The first epoch, seed 0, each optimizer with its best η",
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3, which="both")
fmt_axes(fig)
plt.show()

After 1 epoch the three curves are still close together and still going down. One seed and one epoch can't tell yet
which optimizer is better. The rest of the race (5 epochs, 3 seeds, the whole $\eta$ grid, two schedules) takes a few
minutes, so we read the results the demo saved.

## 7 · The full results: 3 seeds, the whole $\eta$ grid

For each optimizer and each schedule, the table below takes the $\eta$ with the lowest mean validation loss. The
range in parentheses is the min–max over the 3 seeds. The test-set accuracy is only for reporting; it isn't used to
choose anything.

In [ ]:
def seed_range(r, key="valLoss", d=4):
    v = r[key]
    return f"{num(np.mean(v), d)} ({num(min(v), d)}–{num(max(v), d)})"


for sch, label in [("fixed", "fixed η"), ("cosine", "warmup 5% + cosine")]:
    print(label)
    for opt, name in NAMES.items():
        r = best[(opt, sch)]
        print(
            f"  {name:>8}  η = {num_eta(r['eta']):<5}  validation loss {seed_range(r)}  "
            f"validation accuracy {pct(np.mean(r['valAcc']))}  test {pct(np.mean(r['testAcc']))}"
        )
    rows = [best[(o, sch)] for o in NAMES]
    overlap = max(min(r["valLoss"]) for r in rows) <= min(max(r["valLoss"]) for r in rows)
    print(f"  the 3-seed ranges of all three overlap: {overlap}")
gain = [max(best[(o, "cosine")]["valLoss"]) < min(best[(o, "fixed")]["valLoss"]) for o in NAMES]
stuck = next(r for r in runs["sweep"] if r["opt"] == "momentum" and r["schedule"] == "fixed" and r["eta"] == 0.3)
print(f"Momentum, fixed η = 0.3: validation accuracy {' · '.join(pct(a) for a in stuck['valAcc'])} (3 seeds)")
print(f"warmup + cosine beats a fixed η in all 3 seeds, for each optimizer: {gain}")

With a fixed $\eta$, all three end up at about the same place: their 3-seed ranges overlap, that is, the differences
lie **within the variation between seeds**. On this small network, SGD with a carefully tuned $\eta$ does no worse
than Adam. This agrees with Choi et al. (2019): comparisons between optimizers depend heavily on how carefully the
hyperparameters are tuned.

Switching to warmup + cosine clearly improves all three, beyond the variation between seeds, while the differences
among the three still lie within it. On this problem, **the learning-rate schedule matters more than the choice of
optimizer**.

The figure below plots the whole grid: the validation loss (the mean of 3 seeds, the bars are the min–max) against
$\eta$, log scale on both axes. The filled points are the best $\eta$; the points marked “✗” at the top are the
values of $\eta$ that made at least one seed blow up.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (sch, title) in zip(axes, [("fixed", "Fixed η"), ("cosine", "Warmup 5% + cosine")]):
    for opt, name in NAMES.items():
        st = STYLE[name]
        rows = [r for r in runs["sweep"] if r["opt"] == opt and r["schedule"] == sch]
        ok = [r for r in rows if r["valLoss"]]
        e = np.array([r["eta"] for r in ok])
        m = np.array([np.mean(r["valLoss"]) for r in ok])
        err = [m - [min(r["valLoss"]) for r in ok], [max(r["valLoss"]) for r in ok] - m]
        ax.errorbar(
            e,
            m,
            yerr=err,
            color=st["color"],
            ls=st["ls"],
            marker=st["marker"],
            mfc="white",
            ms=6,
            lw=1.4,
            capsize=3,
            label=name,
        )
        b = best[(opt, sch)]
        ax.plot(b["eta"], np.mean(b["valLoss"]), st["marker"], color=st["color"], ms=8)
        for r in rows:
            if not r["valLoss"]:
                ax.plot(r["eta"], 2.6, st["marker"], color=st["color"], mfc="white", ms=6)
                ax.annotate(
                    f"✗ {r['diverged']}/3",
                    (r["eta"], 2.6),
                    xytext=(0, -14),
                    textcoords="offset points",
                    ha="center",
                    fontsize=8,
                )
    ax.set(xscale="log", yscale="log", ylim=(0.05, 3.2), xlabel="η (log scale)", title=title)
    ax.grid(alpha=0.3, which="both")
axes[0].set_ylabel("validation loss after 5 epochs")
axes[0].legend(frameon=False, fontsize=9, loc="upper left")
fmt_axes(fig)
plt.tight_layout()
plt.show()

Each optimizer has its own “valley” of $\eta$, and their scales are offset from one another: SGD around 0.3–1,
Momentum 10 times smaller, Adam around 0.001–0.003. Both sides of the valley are steep. With $\eta$ too small, the
network still hasn't finished learning after 5 epochs; with $\eta$ too large, it blows up (SGD $\eta = 3$) or gets
stuck at a high loss (Momentum $\eta = 0.3$: 9.66–30.04% correct, close to random guessing).

The per-epoch curves of the three best runs (seed 0, also from the demo's file; the last point is after 5 epochs):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.3), sharey=True)
for ax, (sch, title) in zip(axes, [("fixed", "Fixed η"), ("cosine", "Warmup 5% + cosine")]):
    for opt, name in NAMES.items():
        c = next(c for c in runs["curves"] if c["opt"] == opt and c["schedule"] == sch)
        st = STYLE[name]
        ax.plot(
            np.arange(len(c["val"])) / 4,
            c["val"],
            color=st["color"],
            ls=st["ls"],
            marker=st["marker"],
            ms=3.5,
            lw=1.5,
            label=f"{name}, η = {num_eta(c['eta'])}",
        )
    ax.set(yscale="log", ylim=(0.05, 3), xlabel="epoch", title=title)
    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("validation loss")
fmt_axes(fig)
plt.tight_layout()
plt.show()
print(
    "final validation loss (seed 0):",
    " · ".join(f"{c['opt']}/{c['schedule']} {num(c['val'][-1], 4)}" for c in runs["curves"]),
)

## 8 · Warmup and the learning-rate decay schedule

The demo's “warmup + cosine” schedule (also the form used by LLaMA and nanoGPT):
- **warmup:** over the first 5% of the steps (215 steps), $\eta$ rises linearly from 0 to the peak $\eta$;
- **cosine:** after that, $\eta$ decreases along half a cosine period, down to 10% of the peak $\eta$ at the last
  step.

In the table in section 7, SGD with a fixed $\eta = 1$ blows up in 1 of the 3 seeds, while with warmup $\eta = 1$ is
actually the best $\eta$. The cell below reruns the first 215 steps (exactly the warmup, ¼ epoch) of both schedules
for 3 seeds. Within each seed, the two schedules start from the same weights and see the same batches.

In [ ]:
w_steps = round(runs["warmup"] * STEPS)
print(
    f"warmup {pct(runs['warmup'], 0)} = {w_steps} steps; cosine down to {pct(runs['floor'], 0)} of the peak η at step {num(STEPS, 0)}"
)
warm = {}
for seed in range(3):
    for sch in ["fixed", "cosine"]:
        _, log, _ = train(sgd, 1.0, sch, seed=seed, n=215)
        warm[(seed, sch)] = log
        L = np.array(log["loss"])
        end = (
            f"blew up at step {log['boom']}"
            if log["boom"]
            else f"after 215 steps: validation {num(log['val'][-1][0], 3)}"
        )
        print(
            f"seed {seed}, {('fixed η = 1' if sch == 'fixed' else 'η = 1 + warmup'):<14}: "
            f"highest batch loss {num(L[np.isfinite(L)].max(), 1):>5} · {end}"
        )
row = next(r for r in runs["sweep"] if r["opt"] == "sgd" and r["schedule"] == "fixed" and r["eta"] == 1)
n_boom = sum(warm[(s, "fixed")]["boom"] is not None for s in range(3))
print(f"seeds that blow up with a fixed η = 1: {n_boom}/3 (demo, after 5 epochs: {row['diverged']}/3)")
assert warm[(2, "fixed")]["boom"] == 8 and n_boom == row["diverged"], "differs from the demo: check the data"
assert all(warm[(s, "cosine")]["boom"] is None for s in range(3))

The left plot shows the two schedules, $\eta(t)/\eta_{\text{peak}}$ against the epoch. The right plot shows the loss
of each batch over the first 100 steps (log scale): the fixed $\eta$ is the dashed orange line, the one with warmup
the solid blue line; the marker shapes distinguish the 3 seeds, and the black × marks where a run blew up.

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.3), gridspec_kw={"width_ratios": [1, 1.4]})
t_all = np.arange(1, STEPS + 1)
a.plot(t_all / 860, [lr_at(1, "fixed", t) for t in t_all], color=ORANGE, ls=(0, (6, 3)), lw=2, label="fixed")
a.plot(t_all / 860, [lr_at(1, "cosine", t) for t in t_all], color=BLUE, lw=2, label="warmup 5% + cosine")
a.set(xlabel="epoch", ylabel="η(t) / peak η", title="Two learning-rate schedules", ylim=(0, 1.08))
a.legend(frameon=False, fontsize=9)
for seed, mk in zip(range(3), ["o", "s", "^"]):
    for sch, color, ls in [("fixed", ORANGE, (0, (6, 3))), ("cosine", BLUE, "-")]:
        L = np.array(warm[(seed, sch)]["loss"][:100])
        k = np.arange(1, len(L) + 1)
        ok = np.isfinite(L)
        b.plot(k[ok], L[ok], color=color, ls=ls, lw=1.2, marker=mk, ms=4, markevery=10, label=f"seed {seed}")
        if not ok.all():
            b.plot(k[ok][-1], L[ok][-1], "x", color="black", ms=10, mew=2)
            b.annotate("blew up", (k[ok][-1], L[ok][-1]), xytext=(6, 0), textcoords="offset points", va="center")
b.set(yscale="log", xlabel="step", ylabel="batch loss", title="SGD, η = 1: the first 100 steps")
handles, labels = b.get_legend_handles_labels()
b.legend(handles[1::2], labels[1::2], frameon=False, fontsize=9, title="blue: warmup · orange: fixed", title_fontsize=9)
fmt_axes(fig)
plt.tight_layout()
plt.show()

With $\eta = 1$ from the very first step, the loss of a batch shoots up to 8.3 / 54.8 / 32.3 in the 3 seeds (about
2.4 at the start), and seed 2 blows up at step 8. With warmup, the highest batch loss is only 2.3–2.4, on a par with
the starting level, and it decreases steadily from there.

Why does warmup help? In section 3, SGD is stable when $\eta$ is smaller than 2 divided by the largest curvature.
Kalra & Barkeshli (2024) measure that curvature during training and find that the main benefit of warmup is to allow
a larger peak $\eta$: the small first steps take the network to a region of smaller curvature, where a large $\eta$
becomes stable. This notebook doesn't measure the curvature; we only see its effect.

Take the very seed that just blew up, again with $\eta = 1$, and let warmup + cosine run the full 5 epochs. This is
one cell of the demo's table, and we check that rerunning it gives the same numbers:

In [ ]:
params_w, log_w, _ = train(sgd, 1.0, "cosine", seed=2)
val_loss, val_acc = log_w["val"][-1]
test_acc = evaluate(params_w, Xte, yte)[1]
row = next(r for r in runs["sweep"] if r["opt"] == "sgd" and r["schedule"] == "cosine" and r["eta"] == 1)
print(
    f"SGD, η = 1, warmup + cosine, seed 2, 5 epochs: validation {num(val_loss, 4)} ({pct(val_acc)} correct), "
    f"test {pct(test_acc)} correct"
)
print(
    f"demo, same cell: validation {num(row['valLoss'][2], 4)} ({pct(row['valAcc'][2])} correct), test {pct(row['testAcc'][2])} correct"
)
assert abs(val_loss - row["valLoss"][2]) < 5e-4 and abs(val_acc - row["valAcc"][2]) < 0.001, "differs from the demo"
assert abs(test_acc - row["testAcc"][2]) < 0.001, "differs from the demo: check the data"

The same initial weights, the same batches, the same peak $\eta$: one schedule blows up at step 8, the other belongs
to SGD's best configuration.

The decay at the end helps too. As $\eta$ gets smaller, the mini-batch noise gets smaller with it, so the last steps
fine-tune around the bottom instead of bouncing around it. Large language models use exactly these schedules:
- LLaMA: 2,000 warmup steps, then cosine down to 10% of the peak $\eta$ (Touvron et al., 2023, section 2.3);
- Kimi K2: the WSD schedule (warmup – hold – decay), 500 warmup steps, $\eta$ held at $2 \cdot 10^{-4}$ for the
  first 10 trillion tokens, then cosine down to $2 \cdot 10^{-5}$ (Kimi Team, 2025).

## 9 · AdamW: decoupled weight decay

**Weight decay** shrinks the weights toward 0 a little after every step, so that the network doesn't rely on overly
large weights (Lesson 08 will explain why this helps). There are two ways to write it:

- **L2 penalty:** add $\lambda\theta$ to the gradient, that is, $g \leftarrow g + \lambda\theta$, then feed it to the
  optimizer.
- **Decoupled:** shrink directly, $\theta \leftarrow \theta - \eta\lambda\theta$, outside the optimizer.

With SGD, the two are the same: an SGD step on $g + \lambda\theta$ is exactly the shrinking plus an ordinary SGD
step. Loshchilov & Hutter (2019) showed that with Adam they are not. The $\lambda\theta$ part also gets divided by
$\sqrt{\hat v}$, so weights with large gradients are barely penalized. AdamW takes the shrinking out of Adam:

$$\theta \leftarrow \theta - \eta\left(\frac{\hat m}{\sqrt{\hat v} + \epsilon} + \lambda\theta\right)$$

(This is PyTorch's form, with $\lambda$ multiplied by $\eta$; the original paper multiplies $\lambda$ by the
schedule multiplier, not by $\eta$. The $\lambda$ here is also different from the $\lambda$ of the L2 penalty in
Lesson 08.)

We try it on the same MNIST network. Some pixels at the edge are **never part of a stroke** in the training set, so
the corresponding rows of $W_1$ never receive a gradient from the data. Without weight decay, Adam leaves them
alone: $m = v = 0$, so the step is $0/(0 + \epsilon) = 0$. Weight decay is applied only to the $W$ matrices, not to
the biases $b$, as is usual.

In [ ]:
def adam_l2(theta, g, s, eta, t, lam=0.0):
    adam(theta, g + lam * theta, s, eta, t)  # L2 penalty: λθ goes into the gradient, so Adam divides it by √v̂ too


def adamw(theta, g, s, eta, t, lam=0.0):
    theta -= eta * lam * theta  # decoupled weight decay: shrink every weight by the same factor
    adam(theta, g, s, eta, t)


N_DECAY = 430  # ½ epoch
dead = (Xtr == 0).all(axis=0)  # pixels that are blank in every training image
W1_init = np.random.default_rng(0).normal(0, np.sqrt(2 / 784), (784, 128))  # exactly the initial W1 of train(seed=0)
params_base, _, states_base = train(adam, 1e-3, n=N_DECAY)  # Adam η = 0.001, seed 0, no weight decay
print(f"{dead.sum()} pixels are never part of a stroke → {num(dead.sum() * 128, 0)} weights of W1 receive no gradient")
same = np.array_equal(params_base[0][0][dead], W1_init[dead])
print(f"no weight decay, after {N_DECAY} Adam steps: those weights are still exactly as they started: {same}")
assert dead.sum() == 67 and np.array_equal(params_base[0][0][dead], W1_init[dead])

### 🤔 Predict first
Add an L2 penalty to Adam ($\eta = 0.001$, ½ epoch). With $\lambda = 0.01$ and with $\lambda = 0.0001$ (100 times
smaller), how much do these “no-stroke” weights shrink? And with AdamW, $\lambda = 0.1$?

In [ ]:
decay = {
    "Adam + L2, λ = 0.01": partial(adam_l2, lam=1e-2),
    "Adam + L2, λ = 0.0001": partial(adam_l2, lam=1e-4),
    "AdamW, λ = 0.1": partial(adamw, lam=0.1),
}
dec_runs = {}
w0 = np.abs(W1_init[dead])
print(f"at the start: mean |w| {num(w0.mean(), 4)}, largest {num(w0.max(), 4)}")
for label, rule in decay.items():
    params, log, _ = train(rule, 1e-3, n=N_DECAY, step_bias=adam)
    dec_runs[label] = params
    frac = np.abs(params[0][0][dead]) / w0  # the fraction of each weight that is left, relative to the start
    print(
        f"{label:>22}: |w| left, median {pct(np.median(frac))}, largest {pct(frac.max())} of the initial value"
        f" · validation {num(log['val'][-1][0], 4)}"
    )
print(f"AdamW: (1 − η·λ)^{N_DECAY} = (1 − 0.001·0.1)^{N_DECAY} = {num((1 - 1e-3 * 0.1) ** N_DECAY, 4)}")
print(
    f"with L2, each step is ≈ η long: the median |w| {num(np.median(w0), 3)} needs about {num(np.median(w0) / 1e-3, 0)} steps to reach 0"
)
medians = [np.median(np.abs(dec_runs[k][0][0][dead]) / w0) for k in list(decay)[:2]]
assert max(medians) < 1e-6  # L2 penalty: almost down to 0, with both values of λ
assert np.allclose(dec_runs["AdamW, λ = 0.1"][0][0][dead], W1_init[dead] * (1 - 1e-4) ** N_DECAY)

With the L2 penalty, both values of $\lambda$ wipe those weights out almost completely: λ is 100 times smaller and the
result doesn't change. The reason is that their gradient is now just $\lambda\theta$, and Adam divides it by its own
magnitude, so each step is about $\eta$ long **regardless of whether $\lambda$ is large or small**. After about
$|\theta|/\eta$ steps (a few dozen steps) the weight reaches 0. With AdamW, every one of these weights shrinks by
exactly $(1 - \eta\lambda)$ per step, matching the formula to the last digit.

What about the weights that *do* have gradients? The figure below divides the remaining weights of $W_1$ into 8 equal
groups by gradient magnitude $\sqrt{\hat v}$ in the run without weight decay (left: small, right: large). Each point
is the median of $|w| / |w_{\text{no decay}}|$ within the group: 1 means no shrinking at all.

In [ ]:
active = np.broadcast_to(~dead[:, None], W1_init.shape)
rms = np.sqrt(states_base[0]["v"] / (1 - 0.999**N_DECAY))[active]  # the gradient magnitude of each weight
groups = np.array_split(np.argsort(rms, kind="stable"), 8)
base_w = np.abs(params_base[0][0][active])
fig, ax = plt.subplots(figsize=(8, 3.4))
for (label, params), (color, mk, ls) in zip(
    dec_runs.items(), [(ORANGE, "s", (0, (6, 3))), (ORANGE, "D", ":"), (GREEN, "^", "-")]
):
    w = np.abs(params[0][0][active])
    med = [np.median(w[g] / base_w[g]) for g in groups]
    ax.plot(range(1, 9), med, color=color, marker=mk, ls=ls, lw=1.6, ms=7, label=label)
    print(f"{label:>22}: median |w|/|w no decay| by group 1…8: {' · '.join(num(m, 2) for m in med)}")
ax.axhline(1, color=GREY, lw=0.8)
ax.set(
    xlabel="weight groups by gradient magnitude (1 = smallest, 8 = largest)",
    ylabel="|w| / |w no decay|",
    ylim=(-0.03, 1.08),
)
ax.set_title("In Adam, L2 shrinks small-gradient weights much more; AdamW shrinks evenly", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower right")
fmt_axes(fig)
plt.show()

The L2 penalty in Adam shrinks the weights with small gradients the most, and the larger the gradient, the less it
shrinks them: with $\lambda = 0.01$, from 0.00 in group 1 to 0.64 in group 8. That is exactly what Loshchilov &
Hutter describe. How much it shrinks them also depends on $\lambda$ in a hard-to-predict way: a $\lambda$ 100 times
smaller still wipes out group 1. AdamW shrinks every group by almost the same factor, 0.96–0.97, close to the pure
shrinking factor 0.9579 (a little less shrinking, because the gradient from the data partly pulls back).

The practical consequence: with AdamW, $\lambda$ keeps its exact meaning, “how large a fraction to shrink at each
step”, and the best $\lambda$ depends less on $\eta$ (Loshchilov & Hutter, 2019). That is why AdamW is the default
choice for training Transformers:
- LLaMA: AdamW with $\beta_1 = 0.9$, $\beta_2 = 0.95$, $\lambda = 0.1$, gradient clipping at 1.0 (Touvron et al.,
  2023);
- PyTorch's defaults: $\eta = 0.001$, $\beta_2 = 0.999$, $\lambda = 0.01$.

$\beta_2 = 0.95$ gives $\hat v$ a shorter memory (about 20 steps instead of 1,000), so Adam reacts faster when the
gradient magnitude suddenly increases.

## 10 · A new optimizer: Muon

Adam adjusts the step size *element* by element. Muon (Keller Jordan, 2024) looks at the whole *matrix*: it takes
Momentum's velocity $V$ for a weight matrix and replaces $V$ with the nearest matrix whose singular values are all
equal to 1. If $V = USV^\top$ (the SVD), the update is $UV^\top$, that is, every direction in the step has the same
length. Instead of an SVD, Muon uses 5 Newton–Schulz iterations made only of matrix multiplications. We apply it to
the velocity of $W_2$ (the 128×64 hidden matrix) after the 1 epoch of Momentum in section 6:

In [ ]:
def newton_schulz5(G, steps=5):
    """Push every singular value of G close to 1, using only matrix multiplications (Keller Jordan's coefficients,
    2024)."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G / np.linalg.norm(G)
    tall = X.shape[0] > X.shape[1]
    X = X.T if tall else X
    for _ in range(steps):
        A = X @ X.T
        X = a * X + (b * A + c * A @ A) @ X
    return X.T if tall else X


V = live["Momentum"][2][2]["v"]  # the velocity of W2 after 860 steps
s_before = np.linalg.svd(V, compute_uv=False)
s_after = np.linalg.svd(newton_schulz5(V), compute_uv=False)
print(f"velocity of W2: {V.shape[0]}×{V.shape[1]}, {len(s_before)} singular values")
print(
    f"before: from {num(s_before.min(), 4)} to {num(s_before.max(), 3)} (largest / smallest = {num(s_before.max() / s_before.min(), 0)})"
)
print(
    f"after 5 iterations: from {num(s_after.min(), 2)} to {num(s_after.max(), 2)}; in [0.5, 1.3]: {((s_after > 0.5) & (s_after < 1.3)).sum()}/{len(s_after)}"
)

Before, a few directions take up almost the whole update; after 5 iterations, 63/64 directions have a length between
0.5 and 1.3 (not exactly 1: the coefficients are chosen to converge fast, at the cost of some error). Muon is used
only for the hidden matrices; the embeddings, the output layer, the biases and the normalization parameters still
use AdamW. Kimi K2 (1 trillion parameters) was trained with MuonClip, a variant of Muon (Kimi Team, 2025). This
notebook doesn't train with Muon; see Keller Jordan's post if you want to try it.

## 11 · Try changing things

The cells below rerun the same functions with different parameters. You can edit the values and run them again.

### Adam's bias correction, on MNIST
On the valley, dropping the correction only made the first step about 3.16 times longer. On the real network, we run
the first 100 steps with a small $\eta$ and a large $\eta$:

In [ ]:
for eta in [1e-3, 3e-2]:
    for correct in [True, False]:
        params_bc, log, _ = train(partial(adam, correct=correct), eta, n=100)
        print(
            f"Adam η = {num_eta(eta)}, {'with' if correct else 'without'} correction: highest batch loss "
            f"{num(max(log['loss']), 1):>4}, validation after 100 steps {num(evaluate(params_bc, Xval, yval)[0], 3)}"
        )

With a small $\eta$, dropping the correction even helps a little: the first steps are about 3 times longer, like a
larger $\eta$. With $\eta = 0.03$, those same long steps make the loss of a batch shoot up to 58.2, and after 100
steps the network is still far behind (validation 1.946 against 0.337). Bias correction keeps the first step from
being longer than $\eta$, whatever $\eta$ is.

### The valley's tilt and Adam's $\eta$
In section 5, a 20° tilt made Adam almost 3 times slower. Change the angle and $\eta$ to see how much Adam depends on
the axes (SGD and Momentum don't depend on them at all):

In [ ]:
for deg in [0, 5, 20, 45]:
    a_ = math.radians(deg)
    line = " · ".join(
        f"η = {num(e)}: {outcome(race(adam, e, start=rot(START, a_), tilt=a_))}" for e in [0.05, 0.2, 0.5]
    )
    print(f"tilted {deg:>2}°  {line}")

### The starting point
Change `start` to start closer to the bottom or farther away. Which optimizer gains the most from starting right on
the valley floor, that is, at $\theta_2 = 0$?

In [ ]:
for start in [(-5.5, 2.0), (-5.5, 0.0), (-2.0, 2.0)]:
    print(
        f"start ({num(start[0], 1)}, {num(start[1], 1)}): "
        + " · ".join(
            f"{n} {outcome(race(s, e, start=start))}"
            for n, (s, e) in {"SGD": (sgd, 0.45), "Momentum": (momentum, 0.45), "Adam": (adam, 0.2)}.items()
        )
    )

Starting right on the floor ($\theta_2 = 0$): SGD still arrives at step 113, because its speed is set by the gentle
direction, while Momentum and Adam are twice as fast as when they start at $\theta_2 = 2$. Starting closer to the
bottom along $\theta_1$, at $(-2,\ 2)$, it is SGD that arrives first. No optimizer wins from every starting point.

## 12 · The limit: better optimization doesn't necessarily mean better predictions

An optimizer does only one thing: lower the loss on the **training data**. The cell below compares the training loss
(the mean of the last 43 batches) with the final validation loss of the 6 seed-0 curves the demo saved.

In [ ]:
for c in runs["curves"]:
    print(
        f"{NAMES[c['opt']]:>8} · {'fixed' if c['schedule'] == 'fixed' else 'warmup + cosine':<15}: "
        f"training {num(c['train'][-1], 4)} · validation {num(c['val'][-1], 4)} · validation / training = "
        f"{num(c['val'][-1] / c['train'][-1], 1)}"
    )

Adam with warmup + cosine pushes the training loss the lowest, to 0.0148, about 3 times lower than SGD with a fixed
$\eta$ (0.0461). But the validation loss only drops from 0.0772 to 0.0649. The validation / training ratio grows from
1.5–1.7 (fixed η) to 2.5–4.4 (warmup + cosine): the network starts to memorize the images it has seen. Better
optimization on the training set doesn't automatically turn into better predictions on new images.
There are problems on which Adam generalizes worse than SGD (Wilson et al., 2017), although when its hyperparameters
are tuned carefully enough, Adam does no worse (Choi et al., 2019).

The weight decay of section 9 is one way to fight memorization. Lesson 08 will measure overfitting systematically,
and add the L2 penalty, dropout and data augmentation.

## Summary
- In a valley with condition number $\kappa = 100$, the steep direction caps SGD's $\eta$ (it bounces when
  $\eta > 0.25$ and blows up when $\eta > 0.5$) and it crawls slowly along the gentle direction: it arrives at step
  113.
- Momentum $v \leftarrow \beta v + g$ accumulates the gradients, the older the step the lighter its weight; it
  arrives at step 69 and tolerates $\eta$ up to $(2 + 2\beta)/4$. The largest $\beta$ is not the best: with
  $\beta = 0.99$ it hasn't arrived after 200 steps.
- Adam divides by $\sqrt{\hat v}$, parameter by parameter; bias correction makes the first step exactly $\eta$ long
  along each axis. It arrives at step 61, but with the valley tilted by 20° that becomes step 168: its advantage is
  tied to the axes.
- On MNIST, Momentum's best η is 10 times smaller than SGD's ($1/(1-\beta)$). With a fixed $\eta$, the three are level
  within the variation between seeds; warmup + cosine takes all three beyond that variation.
- Warmup allows $\eta = 1$: with a fixed η, seed 2 blows up at step 8; with warmup, the same seed belongs to SGD's best
  configuration.
- In Adam, the L2 penalty gets divided by $\sqrt{\hat v}$, so it shrinks the weights unevenly and not in line with
  $\lambda$; AdamW shrinks every weight by $(1 - \eta\lambda)$ per step.

## Review questions
1. In the valley $L = 0.02\,\theta_1^2 + 2\,\theta_2^2$, why is there no $\eta$ with which SGD is both bounce-free
   and fast?
2. Why is the best $\eta$ of Momentum in the form $v \leftarrow \beta v + g$ about 10 times smaller than SGD's when
   $\beta = 0.9$?
3. How long is Adam's first step with bias correction, along each axis? And without the correction?
4. Rotating the valley changes nothing for SGD and Momentum but makes Adam slower. Why?
5. With $\eta = 1$, SGD with a fixed η blows up in seed 2 while with warmup it doesn't, even with the same initial
   weights and the same batches. What does warmup change in the first steps?
6. A weight never receives a gradient from the data. With Adam + an L2 penalty, why does it reach 0 just as fast
   whether $\lambda = 0.01$ or $0.0001$? What does AdamW do with it?

<details><summary>Hints and answers</summary>

1. In the steep direction, each step multiplies $\theta_2$ by $1 - 4\eta$: not bouncing requires $\eta \le 0.25$, not
   blowing up requires $\eta < 0.5$. In the gentle direction, each step multiplies $\theta_1$ by only
   $1 - 0.04\,\eta \ge 0.98$, so with every allowed $\eta$, SGD crawls slowly.
2. Along a direction where the gradient barely changes, $v$ approaches $g/(1-\beta) = 10\,g$, so Momentum with $\eta$
   moves almost like SGD with $10\eta$.
3. Exactly $\eta$: $\hat m_1/\sqrt{\hat v_1} = g/|g| = \pm 1$. Without the correction,
   $m_1/\sqrt{v_1} = 0.1/\sqrt{0.001}$, that is, about 3.16 times $\eta$; on MNIST with $\eta = 0.03$, that makes the
   loss shoot up.
4. SGD and Momentum use only the vector $g$, so the path rotates along with the axes. Adam divides each component by
   its own magnitude, so it only helps when the steep and the gentle directions lie along the parameters.
5. The first steps use a small $\eta$, so no step is large enough to make the loss shoot up; according to Kalra &
   Barkeshli (2024), they take the network to a region of smaller curvature, where a large $\eta$ becomes stable.
6. Its gradient is just $\lambda\theta$, and Adam divides it by its own magnitude, so each step is about $\eta$ long
   regardless of $\lambda$. AdamW multiplies it by $(1 - \eta\lambda)$ at each step, exactly as $\lambda$ prescribes.
</details>

## Further reading
- B. T. Polyak (1964), [Some methods of speeding up the convergence of iteration methods](https://doi.org/10.1016/0041-5553(64)90137-5): “heavy ball” momentum.
- G. Goh (2017), [Why Momentum Really Works](https://distill.pub/2017/momentum/), Distill: momentum on quadratic
  functions, with interactive figures; the source of the threshold $2 + 2\beta$ and of $\beta^*$.
- D. Kingma & J. Ba (2015), [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980): Algorithm 1
  and section 3 on bias correction.
- I. Loshchilov & F. Hutter (2019), [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101): AdamW.
- D. Choi et al. (2019), [On Empirical Comparisons of Optimizers for Deep Learning](https://arxiv.org/abs/1910.05446):
  how optimizers rank depends on how their hyperparameters are tuned.
- A. Wilson et al. (2017), [The Marginal Value of Adaptive Gradient Methods in Machine Learning](https://arxiv.org/abs/1705.08292).
- D. S. Kalra & M. Barkeshli (2024), [Why Warmup the Learning Rate?](https://arxiv.org/abs/2406.09405)
- H. Touvron et al. (2023), [LLaMA](https://arxiv.org/abs/2302.13971), section 2.3: AdamW and the cosine schedule of a
  large language model.
- K. Jordan (2024), [Muon: An optimizer for hidden layers in neural networks](https://kellerjordan.github.io/posts/muon/).
- Kimi Team (2025), [Kimi K2: Open Agentic Intelligence](https://arxiv.org/abs/2507.20534): MuonClip and the WSD schedule.

**Next lesson:** [Lesson 08 · Overfitting & regularization](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/chong-qua-khop/):
when the network memorizes the training data, and how to fight it with the L2 penalty, dropout and data augmentation.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")